# LogitLens Smoke Test

This notebook verifies that the LogitLens workbench module works correctly on Google Colab with NDIF.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davidbau/workbench/blob/main/workbench/logitlens/notebooks/smoke_test.ipynb)

## Prerequisites

Before running, add these secrets to Colab:
1. Click the key icon in Colab's left sidebar
2. Add these secrets (enable "Notebook access" for each):
   - `NDIF_API` - Your key from [nnsight.net](https://nnsight.net)
   - `HF_TOKEN` - Your token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (required for Llama)

In [ ]:
# Install packages from kitwidget branch (includes squeeze fix for nnsight #581)
!pip install -q nnsight "interp-workbench @ git+https://github.com/davidbau/workbench.git@kitwidget"

In [ ]:
# Configure API keys
import os
from nnsight import CONFIG

NDIF_API = None
HF_TOKEN = None

# Try Colab secrets first
try:
    from google.colab import userdata
    NDIF_API = userdata.get('NDIF_API')
    print("Got NDIF_API from Colab secrets")
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
        print("Got HF_TOKEN from Colab secrets")
    except:
        print("HF_TOKEN not found in Colab secrets")
except Exception as e:
    print(f"Colab secrets not available: {e}")

# Fall back to environment variables
if not NDIF_API:
    NDIF_API = os.environ.get('NDIF_API') or os.environ.get('NDIF_API_KEY')
    if NDIF_API:
        print("Got NDIF_API from environment")

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
    if HF_TOKEN:
        print("Got HF_TOKEN from environment")

# Configure NDIF
if NDIF_API:
    CONFIG.set_default_api_key(NDIF_API)
    print("NDIF configured successfully!")
else:
    raise ValueError("No NDIF_API found. Add it to Colab secrets or set NDIF_API environment variable.")

# Configure HuggingFace (for gated models like Llama)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print("HF_TOKEN configured for gated model access")
else:
    print("Warning: HF_TOKEN not set. Gated models (like Llama) may not work.")

In [ ]:
# Test 1: Import check
print("Test 1: Importing workbench.logitlens...")
from workbench.logitlens.collect import collect_logit_lens
from workbench.logitlens.display import show_logit_lens, to_js_format
print("  PASS: Imports successful")

In [ ]:
# Test 2: Model loading
print("Test 2: Loading model...")
from nnsight import LanguageModel
model = LanguageModel("meta-llama/Llama-3.1-8B", device_map="auto")
print(f"  PASS: Loaded {model.config._name_or_path}")
print(f"  Layers: {model.config.num_hidden_layers}")
print(f"  Vocab: {model.config.vocab_size}")

In [ ]:
# Test 3: Data collection
print("Test 3: Collecting logit lens data...")
data = collect_logit_lens("Hello world", model, k=5, remote=True)
print(f"  PASS: Collected data")
print(f"  Input tokens: {data['input']}")
print(f"  Layers: {len(data['layers'])}")
print(f"  TopK shape: {data['topk'].shape}")

In [ ]:
# Test 4: JSON conversion
print("Test 4: Converting to JS format...")
js_data = to_js_format(data)
assert "meta" in js_data
assert "input" in js_data
assert "layers" in js_data
assert "topk" in js_data
assert "tracked" in js_data
print("  PASS: JSON format valid")

In [ ]:
# Test 5: Widget rendering
print("Test 5: Rendering widget...")
html = show_logit_lens(data, title="Smoke Test: Hello world")
assert "LogitLensWidget" in html.data
assert "Hello" in html.data
print("  PASS: Widget HTML generated")
print()
print("=" * 50)
print("ALL TESTS PASSED!")
print("=" * 50)

In [ ]:
# Display the widget to verify visual rendering
show_logit_lens(data, title="Smoke Test: Hello world")

In [ ]:
# Test 6: UI options
print("Test 6: Testing UI options...")
html2 = show_logit_lens(
    data,
    title="With Options",
    dark_mode=True,
    chart_height=200,
    pinned_rows=[]
)
assert "darkMode" in html2.data or '"darkMode":true' in html2.data
print("  PASS: UI options applied")
html2